In [2]:
import numpy as np
import h5py as h5
import matplotlib.pyplot as plt
import astropy.units as u
from typing import Union, Dict, List, Tuple, Optional
from dataclasses import dataclass
from pathlib import Path

import os
import glob
from collections import defaultdict
import pandas as pd

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import sys
sys.path.append('/Users/sashalvna/Research/GROWL-catalog-public/formation_channels')


In [3]:
sys.path.append('/Users/sashalvna/Research/GROWL-catalog-public/growl_common_code') 
from growl_catalog import *
from common_functions import *

### Some notes:
    
based on https://github.com/Adam-Boesky/Exploring_Parameter_Space/blob/master/Data_Analysis/Scripts/formation_channels.py, note that adam had optimistic CE

### To do 

 - remove `create_separate_files=True` option, dont need this & dont want this. I always want to create a seperate hdf5 file for each entry for now
 - clean up more 
 - add other (COMPAS) datasets, and make function in seperate python class/folder as functions will be re-used outside of this notebook 
 - add other DCO parameters (& ZAMS parameters): , SN/CHE, separation (to get spins), eccentricity
 
 
done:
 - added formation channel
 - improved file structure to seperate BNS BHNS and BBH
 - added optimistic files (but should move this into seperate folder... to make it more like a parameter
 
  

In [4]:
def get_COMPAS_vars(compas_file, group, variables, mask=None):
    """Return a variable from the COMPAS output data

    Parameters
    ----------
    input_file : `hdf5 File`
        COMPAS file
    group : `str`
        Group within COMPAS file
    variables : `str` or `list`
        List of column names to access in group (or single column name)
    mask : `bool/array`
        Mask of binaries with same shape as each variable

    Returns
    -------
    var_list : `various`
        Single variable or list of variables (all masked)
    """
    var_list = None
    if isinstance(variables, str):
        var_list = compas_file[group][variables][...].squeeze()
        if mask is not None:
            var_list = var_list[mask]
    else:
        if mask is not None:
            var_list = [compas_file[group][var][...].squeeze()[mask]
                        for var in variables]
        else:
            var_list = [compas_file[group][var][...].squeeze()
                        for var in variables]

    return var_list

In [ ]:
import numpy as np
import collections


@dataclass
class DCOParameters:
    """Container for Double Compact Object parameters"""
    metallicities: np.ndarray
    delay_times: np.ndarray
    formation_efficiencies: np.ndarray
    dco_masses_1: np.ndarray
    dco_masses_2: np.ndarray
    primary_masses: np.ndarray
    secondary_masses: np.ndarray
    chirp_masses: np.ndarray
    mixture_weights: np.ndarray
    total_mass_evolved: float
    n_systems: int
    formation_channel: np.ndarray


        
def get_file_path(catalog, author, dataset):
    """
    Get the full path to an HDF5 file for a specific author and dataset.
    """
    if author not in catalog:
        raise ValueError(f"Author '{author}' not found in catalog")
    
    if dataset not in catalog[author]['paths']:
        raise ValueError(f"Dataset '{dataset}' not found for author '{author}'")
    
    path = catalog[author]['paths'][dataset]
    file_name = catalog[author]['file_name']
    return os.path.join(path, file_name)

def list_authors(catalog):
    """Get list of all authors."""
    return list(catalog.keys())

def list_datasets(catalog, author):
    """Get list of all datasets for a specific author."""
    if author not in catalog:
        raise ValueError(f"Author '{author}' not found in catalog")
    return catalog[author]['datasets']
        

def print_catalog_summary(catalog):
    """Print a summary of the catalog structure."""
    print("GROWL Catalog Summary:")
    print("=" * 50)
    
    for author in sorted(catalog.keys()):
        print(f"\nAuthor: {author}")
        print(f"  File: {catalog[author]['file_name']}")
        print(f"  Datasets ({len(catalog[author]['datasets'])}):")
        for dataset in catalog[author]['datasets']:
            print(f"    - {dataset}")

        

def build_growl_catalog(base_path='/Users/sashalvna/Research/Fit_SFRD_TNG/data/'):
    """
    Build a dictionary structure for GROWL catalog with authors and their datasets.
    
    Structure:
    {
        'author_name': {
            'datasets': ['dataset1', ...],
            'file_name': 'COMPAS_Output_Weighted.h5',
            'paths': {
                'dataset1': '/Volumes/GROWL/GROWL_bps/vanSon22/fiducial/''
            }
            'labels':{'dataset1': r'fiducial'
            
            }
        }
    }
    """
    catalog = {}
    
    if not os.path.exists(base_path):
        print(f"Base path {base_path} does not exist")
        return catalog
    
    # Get all author directories
    author_dirs = [d for d in os.listdir(base_path) 
                  if os.path.isdir(os.path.join(base_path, d)) and not d.startswith('.')]
    
    for author in author_dirs:
        author_path = os.path.join(base_path, author)
        
        # Get all dataset directories for this author
        dataset_dirs = [d for d in os.listdir(author_path) 
                       if os.path.isdir(os.path.join(author_path, d)) and not d.startswith('.')]
        
        if not dataset_dirs:
            continue
            
        # Find the common HDF5 file name by checking the first dataset
        first_dataset_path = os.path.join(author_path, dataset_dirs[0])
        h5_files = glob.glob(os.path.join(first_dataset_path, '*.h5'))
        
        if not h5_files:
            print(f"Warning: No HDF5 files found in {first_dataset_path}")
            continue
            
        # Assume the first HDF5 file is the standard one
        file_name = os.path.basename(h5_files[0])
        
        # Build paths dictionary
        paths = {}
        for dataset in dataset_dirs:
            dataset_path = os.path.join(author_path, dataset)
            # Verify the HDF5 file exists in this dataset
            expected_file = os.path.join(dataset_path, file_name)
            if os.path.exists(expected_file):
                paths[dataset] = dataset_path + '/'
            else:
                print(f"Warning: {expected_file} not found")
        
        catalog[author] = {
            'datasets': sorted(dataset_dirs),
            'file_name': file_name,
            'paths': paths
        }
    
    return catalog


def process_multiple_models(
    catalog: Dict, 
    author: str, 
    datasets: List[str], 
    dco_type: str = 'BBH',
    pessimistic: bool = True,
    merges_hubble: bool = True,
    no_RLOF_post_CE: bool = True,
    CHE_mask: str = 'no CHE'
) -> Dict[str, DCOParameters]:
    """
    Process multiple COMPAS models for comparison.
    
    Parameters
    ----------
    catalog : dict
        GROWL catalog dictionary
    author : str
        Author name
    datasets : list
        List of dataset names to process
    dco_type : str
        Type of DCO to extract : 'BBH', 'BHNS', 'BNS'
    pessimistic: bool
        Assuming Pessimistic Common Envelope CE : True (Pessimistic) or False (Optimistic CE)
    merges_hubble : bool
        mask merging in a Hubble time: True, False
    no_RLOF_post_CE : bool
        mask systems with RLOF immediately after CE (assume these are stellar mergers): True, False
    CHE_mask: str = 'no CHE'
        whether to exclude CHE systems 'no CHE' or exclusively only select CHE systems 'only CHE'
        
    Returns
    -------
    dict
        Dictionary with dataset names as keys and DCOParameters as values
    """
    processor = COMPASDataProcessor()
    results = {}
    
    for dataset in datasets:
        try:
            file_path = catalog[author]['paths'][dataset] + catalog[author]['file_name']
            print(f"Processing {author}/{dataset}...")
            
            data = processor.process_compas_file(file_path, dco_type, pessimistic, merges_hubble, no_RLOF_post_CE, CHE_mask)
            if data is not None:
                results[dataset] = data
                print(f"  Found {data.n_systems} {dco_type} systems")
            else:
                print(f"  No {dco_type} systems found")
                
        except Exception as e:
            print(f"Error processing {author}/{dataset}: {e}")
            continue
    
    return results




class COMPASDataProcessor:
    """Class to process COMPAS HDF5 files and extract DCO properties"""
    
    def __init__(self, solar_metallicity: float = 0.0142):
        self.solar_metallicity = solar_metallicity
        
    def analytical_star_forming_mass_per_binary_using_kroupa_imf(
        self, m1_min: float, m1_max: float, m2_min: float, 
        fbin: float = 1., imf_mass_bounds: List[float] = [0.01, 0.08, 0.5, 200]
    ) -> float:
        """
        Analytical computation of the mass of stars formed per binary star formed
        using the Kroupa IMF.
        
        Parameters
        ----------
        m1_min, m1_max : float
            Primary mass range [Msun]
        m2_min : float  
            Minimum secondary mass [Msun]
        fbin : float
            Binary fraction
        imf_mass_bounds : list
            IMF mass boundaries [Msun]
            
        Returns
        -------
        float
            Mass represented by each binary [Msun]
        """
        m1, m2, m3, m4 = imf_mass_bounds
        
        if m1_min < m3:
            raise ValueError(f"This analytical derivation requires IMF break m3 < m1_min ({m3} !< {m1_min})")
        
        alpha = (-(m4**(-1.3) - m3**(-1.3))/1.3 - 
                (m3**(-0.3) - m2**(-0.3))/(m3*0.3) + 
                (m2**0.7 - m1**0.7)/(m2*m3*0.7))**(-1)
        
        # Average mass of stars
        m_avg = alpha * (-(m4**(-0.3) - m3**(-0.3))/0.3 + 
                        (m3**0.7 - m2**0.7)/(m3*0.7) + 
                        (m2**1.7 - m1**1.7)/(m2*m3*1.7))
        
        # Fraction of binaries that COMPAS simulates
        fint = (-alpha / 1.3 * (m1_max**(-1.3) - m1_min**(-1.3)) + 
                alpha * m2_min / 2.3 * (m1_max**(-2.3) - m1_min**(-2.3)))
        
        # Mass represented by each binary
        m_rep = (1/fint) * m_avg * (1.5 + (1-fbin)/fbin)
        
        return m_rep
    
    def get_dco_mask(self, 
                     fdata: h5.File, 
                     dco_type: str = 'BBH', 
                     pessimistic: bool = True, 
                     merges_hubble: bool = True, 
                     no_RLOF_post_CE: bool = True,
                     CHE_mask: str = 'no CHE'
                    ) -> np.ndarray:
        """
        Create mask for Double Compact Objects of specified type.
        
        Parameters
        ----------
        fdata : h5py.File
            COMPAS HDF5 file
        dco_type : str
            Type of DCO: 'BBH', 'BNS', 'NSBH'
        pessimistic : bool
            pessimistic CE: True, False
        merges_hubble : bool
            mask merging in a Hubble time: True, False
        no_RLOF_post_CE : bool
            mask systems with RLOF immediately after CE (assume these are stellar mergers): True, False
        CHE_mask: str
            whether to exclude CHE systems 'no CHE' or exclusively only select CHE systems 'only CHE'
        
        Returns
        -------
        dco_mask : np.ndarray
            Boolean mask for DCOs
        """
        
        print('creating DCO mask for ', dco_type, ' for ', CHE_mask, ' with the following masks: pessimistic - ', pessimistic, \
              ' merges in hubble - ', merges_hubble, ' no RLOF post CE - ', no_RLOF_post_CE )
        
        stellar_type1 = fdata['BSE_Double_Compact_Objects']['Stellar_Type(1)'][()]
        stellar_type2 = fdata['BSE_Double_Compact_Objects']['Stellar_Type(2)'][()]

        
        # Pessimistic CE mask        
        if pessimistic==True:
            optimistic_ce = fdata['BSE_Double_Compact_Objects']['Optimistic_CE'][()].astype(bool)
            
            
            # get a mask that returns 1 if the DCO SEED is NOT in the BSE_CE SEEDS when masking the optimistic CE
            pessimistic_ce_mask = np.in1d(
                fdata['BSE_Double_Compact_Objects']['SEED'][()], 
                fdata['BSE_Double_Compact_Objects']['SEED'][()][optimistic_ce], invert=True
            )
        else: 
            pessimistic_ce_mask = np.repeat(True, len(stellar_type2))
        

        
        # create mask for merges in a Hubble time
        if merges_hubble == True: merges_hubble_mask = (fdata['BSE_Double_Compact_Objects']['Merges_Hubble_Time'][()]==1)
        else: merges_hubble_mask = np.repeat(True, len(stellar_type2))   
        
        # create mask for no RLOF post CE 
        if no_RLOF_post_CE==True:
            rlof_post_ce = fdata['BSE_Double_Compact_Objects']["Immediate_RLOF>CE"][()]
            no_rlof_post_ce_mask = np.in1d(
                fdata['BSE_Double_Compact_Objects']['SEED'][()], 
                fdata['BSE_Double_Compact_Objects']['SEED'][()][rlof_post_ce == 0]
            )
        else: no_rlof_post_ce_mask = np.repeat(True, len(stellar_type2))
        

        # create mask of CHE systems
        che_mask  = np.logical_and.reduce((fdata["BSE_System_Parameters"]["Stellar_Type@ZAMS(1)"][()]==16, fdata["BSE_System_Parameters"]["Stellar_Type@ZAMS(2)"][()]==16))
        dco_che_mask = np.in1d(fdata['BSE_Double_Compact_Objects']['SEED'][()], fdata["BSE_System_Parameters"]["SEED"][()][che_mask])
        if CHE_mask=='no CHE': mask_che_dco = (dco_che_mask==0) # return True for systems that do NOT have CHE 
        elif CHE_mask=='only CHE': mask_che_dco = (dco_che_mask==1) # return True only for systems that DO have CHE 

     
        
        # Define stellar type mappings
        type_map = {'NS': 13, 'BH': 14}
        
        if dco_type == 'BBH':
            type_mask = (stellar_type1 == type_map['BH']) & (stellar_type2 == type_map['BH']) 
        elif dco_type == 'BNS':
            type_mask = (stellar_type1 == type_map['NS']) & (stellar_type2 == type_map['NS'])
        elif (dco_type == 'NSBH') | (dco_type == 'BHNS'):
            type_mask = ((stellar_type1 == type_map['NS']) & (stellar_type2 == type_map['BH'])) | \
                       ((stellar_type1 == type_map['BH']) & (stellar_type2 == type_map['NS']))
        else:
            raise ValueError(f"Unknown DCO type: {dco_type}")
        
    

        
        dco_mask = type_mask & (merges_hubble_mask == True) & (pessimistic_ce_mask == True) & (no_rlof_post_ce_mask == True) & (mask_che_dco == True)
        
        if CHE_mask=='no CHE':
            CHE_dco_mask = type_mask & (merges_hubble_mask == True) & (pessimistic_ce_mask == True) & (no_rlof_post_ce_mask == True) & (dco_che_mask == 1)
            print('FYI: the original datafile has %s CHE DCO systems of interest, these are not included but should be run with CHE_mask = only CHE'%np.sum(CHE_dco_mask))
        
        
        return dco_mask
    
    
    def get_primary_secondary(self, m1: np.ndarray, m2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Return (primary, secondary) where primary >= secondary element-wise.
        
        Parameters
        ----------
        m1, m2 : np.ndarray
            Component masses
            
        Returns
        -------
        primary, secondary : np.ndarray
            Ordered masses
        """
        primary = np.maximum(m1, m2)
        secondary = np.minimum(m1, m2)
        return primary, secondary
    
    def chirp_mass(self, m1: np.ndarray, m2: np.ndarray) -> np.ndarray:
        """
        Compute chirp mass from component masses.
        
        Parameters
        ----------
        m1, m2 : np.ndarray
            Component masses [Msun]
            
        Returns
        -------
        np.ndarray
            Chirp masses [Msun]
        """
        return (m1 * m2)**(3/5) / (m1 + m2)**(1/5)
    
    def process_compas_file(self, file_path: str,
                            dco_type: str = 'BBH', 
                            pessimistic: bool = True,
                            merges_hubble: bool = True, 
                            no_RLOF_post_CE: bool = True,
                            CHE_mask: str = 'no CHE'
                           ) -> DCOParameters:
        """
        Process a COMPAS HDF5 file and extract DCO parameters.
        
        Parameters
        ----------
        file_path : str
            Path to COMPAS HDF5 file
        dco_type : str
            Type of DCO to extract
        pessimistic : bool
            pessimistic CE: True, False
        merges_hubble : bool
            mask merging in a Hubble time: True, False
        no_RLOF_post_CE : bool
            mask systems with RLOF immediately after CE (assume these are stellar mergers): True, False
        CHE_mask: str = 'no CHE'
            whether to exclude CHE systems 'no CHE' or exclusively only select CHE systems 'only CHE'
            
        Returns
        -------
        DCOParameters
            Container with all DCO properties
        """
        
        
        print('path =', file_path)
        with h5.File(file_path, 'r') as fdata:
            # Get simulation parameters
            initial_mass_min = 10
            initial_mass_max = 150 
            minimum_secondary_mass = 0.1
            
            # Calculate mass representation
            m_rep_per_binary = self.analytical_star_forming_mass_per_binary_using_kroupa_imf(
                m1_min=initial_mass_min, 
                m1_max=initial_mass_max,
                m2_min=minimum_secondary_mass, 
                fbin=1.0
            )
            
            n_binaries = len(fdata['BSE_System_Parameters']['SEED'][()])
            total_mass_evolved = n_binaries * m_rep_per_binary
            
            # Get DCO mask
            dco_mask = self.get_dco_mask(fdata, dco_type, pessimistic, merges_hubble, no_RLOF_post_CE, CHE_mask)
            n_dcos = np.sum(dco_mask)
            
            if n_dcos == 0:
                print(f"Warning: No {dco_type} systems found in {file_path}")
                return None
            
            # Get system parameters for DCOs
            mask_sys_dcos = np.in1d(
                fdata['BSE_System_Parameters']['SEED'][()], 
                fdata['BSE_Double_Compact_Objects']['SEED'][()][dco_mask]
            )
            
            # Extract properties
            metallicities = fdata['BSE_System_Parameters']['Metallicity@ZAMS(1)'][()][mask_sys_dcos]
            mixture_weights = fdata['BSE_Double_Compact_Objects']['mixture_weight'][()][dco_mask]
            #formation_efficiencies = mixture_weights / total_mass_evolved #the old way - with sspc, gives low rates

            #from alex's code in population_vis.py 
            dNdco, bins = np.histogram(np.log10(metallicities), bins=50, density=True, weights=mixture_weights) 
            formation_efficiencies_binned = dNdco*np.sum(mixture_weights/total_mass_evolved)
            formation_efficiencies = formation_efficiencies_binned[np.digitize(np.log10(metallicities), (bins[:-1]+bins[1:])/2) - 1]
            
            delay_times = (fdata['BSE_Double_Compact_Objects']['Coalescence_Time'][()] + 
                          fdata['BSE_Double_Compact_Objects']['Time'][()])[dco_mask]
            
            # Masses
            dco_masses_1 = fdata['BSE_Double_Compact_Objects']['Mass(1)'][()][dco_mask]
            dco_masses_2 = fdata['BSE_Double_Compact_Objects']['Mass(2)'][()][dco_mask]
            primary_masses, secondary_masses = self.get_primary_secondary(dco_masses_1, dco_masses_2)
            chirp_masses = self.chirp_mass(dco_masses_1, dco_masses_2)
            
            # formation channels (fc)
            print('calculating the formation channels for this dataset...')
            seeds_fc = fdata['BSE_Double_Compact_Objects']['SEED'][()][dco_mask] # get seeds of systems we want to calculate fc for

            all_seeds_RLOF = get_COMPAS_vars(fdata, "BSE_RLOF", "SEED")
            mask_RLOF = np.isin(all_seeds_RLOF, seeds_fc)
            masked_seeds_RLOF = all_seeds_RLOF[mask_RLOF]

            all_seeds_System_Parameters = get_COMPAS_vars(fdata, 'BSE_System_Parameters', "SEED")
            mask_System_Parameters = np.isin(all_seeds_System_Parameters, seeds_fc)
            masked_seeds_System_Parameters = all_seeds_System_Parameters[mask_System_Parameters]

            all_seeds_DCO = get_COMPAS_vars(fdata, 'BSE_Double_Compact_Objects', "SEED")
            mask_DCO = np.isin(all_seeds_DCO, seeds_fc)
            masked_seeds_DCO = all_seeds_DCO[mask_DCO]

            count= get_COMPAS_vars(fdata,
                                                                "BSE_RLOF",
                                                                "MT_Event_Counter",
                                                                mask_RLOF)
            stellar_type_1_ZAMS, stellar_type_2_ZAMS = get_COMPAS_vars(fdata,
                                                                'BSE_System_Parameters',
                                                                ['Stellar_Type@ZAMS(1)',
                                                                'Stellar_Type@ZAMS(2)'],
                                                                mask_System_Parameters)
            CE_counter = get_COMPAS_vars(fdata,
                                                                'BSE_Double_Compact_Objects',
                                                                'CE_Event_Counter',
                                                                mask_DCO)
            
            # Only stable channel
            OS_MT_seeds_1 = (count>0) #at least one mass transfer
            OS_MT_seeds_2 = (CE_counter == 0) #no CE
            OS_MT_seeds_3 = (stellar_type_1_ZAMS != 16) #no CHE
            OS_MT_seeds = np.intersect1d(masked_seeds_RLOF[OS_MT_seeds_1],
                                   masked_seeds_DCO[OS_MT_seeds_2])
            OS_MT_seeds_noCHE = np.intersect1d(OS_MT_seeds,
                                   masked_seeds_System_Parameters[OS_MT_seeds_3])
            OS_MT_mask = np.isin(seeds_fc, OS_MT_seeds_noCHE)

            # CEE channel
            CE_seeds_1 = (CE_counter > 0)
            CE_seeds_2 = (stellar_type_1_ZAMS != 16)
            CE_seeds_noCHE = np.intersect1d(masked_seeds_DCO[CE_seeds_1],
                                   masked_seeds_System_Parameters[CE_seeds_2])
            CE_mask = np.isin(seeds_fc, CE_seeds_noCHE)

            # Only stable CHE channel
            CHE_seeds = np.logical_and.reduce(stellar_type_1_ZAMS == 16)
            CHE_OS_MT_seeds = np.intersect1d(OS_MT_seeds,
                                   all_seeds_System_Parameters[CHE_seeds])
            CHE_OS_MT_mask = np.isin(seeds_fc, CHE_OS_MT_seeds)

            # CE CHE channel
            CHE_CE_seeds = np.intersect1d(CE_seeds_1,
                                   all_seeds_System_Parameters[CHE_seeds])
            CHE_CE_mask = np.isin(seeds_fc, CHE_CE_seeds)

            # No mass transfer or CE 
            noMT_seeds_1 = (count == 0)
            noMT_seeds = np.intersect1d(masked_seeds_RLOF[noMT_seeds_1],
                                   masked_seeds_DCO[OS_MT_seeds_2])
            noMT_seeds_noCHE = np.intersect1d(noMT_seeds,
                                   masked_seeds_System_Parameters[OS_MT_seeds_3])
            noMT_mask = np.isin(seeds_fc, noMT_seeds_noCHE)

            channels = np.zeros_like(seeds_fc).astype(int)
            channels[OS_MT_mask] = 1
            channels[CE_mask] = 2
            channels[CHE_OS_MT_mask] = 3
            channels[CHE_CE_mask] = 4
            channels[noMT_mask] = -1
            
            formation_channel = channels

            
        return DCOParameters(
            metallicities=metallicities,
            delay_times=delay_times,
            formation_efficiencies=formation_efficiencies,
            dco_masses_1=dco_masses_1,
            dco_masses_2=dco_masses_2,
            primary_masses=primary_masses,
            secondary_masses=secondary_masses,
            chirp_masses=chirp_masses,
            mixture_weights=mixture_weights,
            total_mass_evolved=total_mass_evolved,
            n_systems=n_dcos,
            formation_channel = formation_channel
        )
    
    



def create_convolution_hdf5_from_dco_data(
    dco_data: Dict[str, DCOParameters], 
    base_output_dir: str,
    author_name: str = "VanSon22",
    output_filename: str = "output_example.h5",
    create_separate_files: bool = False,
    artificial_model_addition=False
) -> Dict[str, str]:
    """
    Create HDF5 file(s) for convolution analysis from processed COMPAS DCO data.
    Creates directory structure: base_output_dir/author_name/dataset_name/
    
    Parameters
    ----------
    dco_data : Dict[str, DCOParameters]
        Dictionary with dataset names as keys and DCOParameters as values
        (e.g., result from process_multiple_models)
    base_output_dir : str
        Base directory (e.g., '/Volumes/GROWL/GROWL_bps_compact')
    author_name : str
        Author directory name (e.g., 'Boesky24')
    output_filename : str
        Name of the output HDF5 file
    create_separate_files : bool
        If True, creates separate HDF5 files for each dataset.
        If False, creates one HDF5 file with all datasets.
    artificial_model_addition : str 
        If False nothing happens, if optimisticCE, we create additional models for optimistic CE
    Returns
    -------
    Dict[str, str]
        Dictionary mapping dataset names to their HDF5 file paths
    """
    output_files = {}
    
    
    
    if create_separate_files:
        for dataset_name, dco_params in dco_data.items():
            
            if artificial_model_addition=='optimisticCE': dataset_name=dataset_name+"_optimisticCE"
            
            output_dir = os.path.join(base_output_dir, author_name, dataset_name)
            os.makedirs(output_dir, exist_ok=True)
            
            output_hdf5_filename = os.path.join(output_dir, output_filename)
            print(f"Processing {dataset_name} -> {output_dir}")
            
            # Prepare the data dictionary with the required properties
            data_dict = {
                "delay_time": dco_params.delay_times,
                "metallicity": dco_params.metallicities,
                "formation_efficiency_per_solar_mass": dco_params.formation_efficiencies,
                "dco_mass_1": dco_params.dco_masses_1,
                "dco_mass_2": dco_params.dco_masses_2,
                "formation_channel": dco_params.formation_channel
            }
            
            
            # Save both data + units
            units_dict = {
                "delay_time": "Myr",
                "metallicity": "#",
                "formation_efficiency_per_solar_mass": "1/Msun",
                "dco_mass_1": "Msun",
                "dco_mass_2": "Msun",
                "formation_channel": "#"
            }                
            

            
            # Create DataFrame
            df = pd.DataFrame.from_records(data_dict)
            
            # Save to HDF5 under a single clean group
            df.to_hdf(output_hdf5_filename, key="input_data", mode='w')  
            pd.Series(units_dict).to_hdf(output_hdf5_filename, key="units", mode="a")   

            print(f"  Saved {len(df)} systems with columns: {list(df.columns)}")
            output_files[dataset_name] = output_hdf5_filename
            
            print(f"HDF5 file created: {output_hdf5_filename}")
    
    return output_files




# Convenience function that matches your existing workflow
def create_hdf5_from_vanSon_data(
    data: Dict[str, DCOParameters],
    base_output_dir: str = '/Users/sashalvna/Research/Fit_SFRD_TNG/data/',
    author_name: str = "vanSon22",
    base_filename: str = "bps_output",
    filename_add: str = "fiducial",
    create_separate_files: bool = True,
    dco_type='BBH',
    artificial_model_addition: str = False
) -> Dict[str, str]:
    """
    Create HDF5 file(s) specifically for Boesky data with proper directory structure.
    Creates: base_output_dir/Boesky24/dataset_name/ (if separate files)
    or: base_output_dir/Boesky24/ (if single file)
    
    Parameters
    ----------
    boesky_data : Dict[str, DCOParameters]
        Result from process_multiple_models(growl_catalog, 'Boesky24', datasets_to_process)
    base_output_dir : str
        Base directory path (e.g., '/Volumes/GROWL/GROWL_bps_compact')
    author_name : str
        Author name (e.g., 'Boesky24')
    filename : str
        Output filename
    create_separate_files : bool
        If True, creates separate files in dataset subdirectories
        If False, creates single file in author directory
    artificial_model_addition: 
        if "False" author name output , if "optimisticCE" create new directory with optimisticCE
    Returns
    -------
    Dict[str, str]
        Dictionary mapping dataset names to HDF5 file paths
    """
    
    if filename_add==False: filename = base_filename + ".h5"
    else: filename = base_filename + filename_add + ".h5"

    return create_convolution_hdf5_from_dco_data(
        dco_data=data,
        base_output_dir=base_output_dir,
        author_name=author_name,
        output_filename=filename,
        create_separate_files=create_separate_files,
        artificial_model_addition=artificial_model_addition
    )




In [6]:
# Build the catalog
growl_catalog = build_growl_catalog()

# Print summary
print_catalog_summary(growl_catalog)




if 'vanSon22' in growl_catalog:
    print(f"\nDatasets for vanSon22:")
    for dataset in list_datasets(growl_catalog, 'vanSon22'):
        print(f"  - {dataset}")
        # Get full file path
        file_path = get_file_path(growl_catalog, 'vanSon22', dataset)
        print(f"    Path: {file_path}")

GROWL Catalog Summary:

Author: vanSon22
  File: COMPAS_Output_wWeights.h5
  Datasets (1):
    - fiducial

Datasets for vanSon22:
  - fiducial
    Path: /Users/sashalvna/Research/Fit_SFRD_TNG/data/vanSon22/fiducial/COMPAS_Output_wWeights.h5


In [7]:
# Assuming you have the GROWL catalog from the previous artifact
growl_catalog = build_growl_catalog()
#print GROWL catalog possible entries
print_catalog_summary(growl_catalog)

vanSon_dataset_list =  list_datasets(growl_catalog, 'vanSon22')
print(vanSon_dataset_list)

GROWL Catalog Summary:

Author: vanSon22
  File: COMPAS_Output_wWeights.h5
  Datasets (1):
    - fiducial
['fiducial']


In [8]:
# New directory structure approach:
dco_type='BBH'
BASE_DIR = '/Users/sashalvna/Research/Fit_SFRD_TNG/data/' + dco_type
filename_add = False

# Process multiple vanSon22 models
for dataset in vanSon_dataset_list[:]: 
    vanSon_data = process_multiple_models(growl_catalog, 'vanSon22', [dataset],dco_type=dco_type, pessimistic= True, merges_hubble=True, no_RLOF_post_CE=True, CHE_mask='no CHE')
    # Create separate entries for each dataset
    output_files = create_hdf5_from_vanSon_data(vanSon_data, BASE_DIR, create_separate_files=True, filename_add=filename_add)
    print('\n \n')


Processing vanSon22/fiducial...
path = /Users/sashalvna/Research/Fit_SFRD_TNG/data/vanSon22/fiducial/COMPAS_Output_wWeights.h5
creating DCO mask for  BBH  for  no CHE  with the following masks: pessimistic -  True  merges in hubble -  True  no RLOF post CE -  True
FYI: the original datafile has 3358 CHE DCO systems of interest, these are not included but should be run with CHE_mask = only CHE
[5.46112538e-06 2.79800082e-06 5.44072080e-06 ... 5.41119885e-06
 2.02706087e-06 5.51205025e-06]
calculating the formation channels for this dataset...
  Found 1637192 BBH systems
Processing fiducial -> /Users/sashalvna/Research/Fit_SFRD_TNG/data/BBH/vanSon22/fiducial
  Saved 1637192 systems with columns: ['dco_mass_1', 'dco_mass_2', 'delay_time', 'formation_channel', 'formation_efficiency_per_solar_mass', 'metallicity']
HDF5 file created: /Users/sashalvna/Research/Fit_SFRD_TNG/data/BBH/vanSon22/fiducial/bps_output.h5

 



In [9]:
dco_type='BHNS'
BASE_DIR = '/Users/sashalvna/Research/Fit_SFRD_TNG/data/' + dco_type
filename_add = False
for dataset in vanSon_dataset_list:
    vanSon_data = process_multiple_models(growl_catalog, 'vanSon22', [dataset], dco_type=dco_type, pessimistic= True, merges_hubble=True, no_RLOF_post_CE=True, CHE_mask='no CHE')
    output_files = create_hdf5_from_vanSon_data(vanSon_data, BASE_DIR, create_separate_files=True, filename_add=filename_add)
    print()

    

Processing vanSon22/fiducial...
path = /Users/sashalvna/Research/Fit_SFRD_TNG/data/vanSon22/fiducial/COMPAS_Output_wWeights.h5
creating DCO mask for  BHNS  for  no CHE  with the following masks: pessimistic -  True  merges in hubble -  True  no RLOF post CE -  True
FYI: the original datafile has 0 CHE DCO systems of interest, these are not included but should be run with CHE_mask = only CHE
[5.28855653e-07 3.75171042e-07 1.24263454e-06 ... 2.93396204e-07
 1.00654286e-06 2.93219847e-09]
calculating the formation channels for this dataset...
  Found 192673 BHNS systems
Processing fiducial -> /Users/sashalvna/Research/Fit_SFRD_TNG/data/BHNS/vanSon22/fiducial
  Saved 192673 systems with columns: ['dco_mass_1', 'dco_mass_2', 'delay_time', 'formation_channel', 'formation_efficiency_per_solar_mass', 'metallicity']
HDF5 file created: /Users/sashalvna/Research/Fit_SFRD_TNG/data/BHNS/vanSon22/fiducial/bps_output.h5



In [10]:
dco_type='BNS'
BASE_DIR = '/Users/sashalvna/Research/Fit_SFRD_TNG/data/' + dco_type
filename_add = False #'_' + dco_type + '_pessimistic'
for dataset in vanSon_dataset_list:
    vanSon_data = process_multiple_models(growl_catalog, 'vanSon22', [dataset], dco_type=dco_type, pessimistic= True, merges_hubble=True, no_RLOF_post_CE=True, CHE_mask='no CHE')
    output_files = create_hdf5_from_vanSon_data(vanSon_data, BASE_DIR, create_separate_files=True, filename_add=filename_add)
    print()

    

Processing vanSon22/fiducial...
path = /Users/sashalvna/Research/Fit_SFRD_TNG/data/vanSon22/fiducial/COMPAS_Output_wWeights.h5
creating DCO mask for  BNS  for  no CHE  with the following masks: pessimistic -  True  merges in hubble -  True  no RLOF post CE -  True
FYI: the original datafile has 0 CHE DCO systems of interest, these are not included but should be run with CHE_mask = only CHE
[1.85194178e-06 1.73764126e-06 9.96651740e-07 ... 1.73764126e-06
 4.09510407e-07 6.16096198e-07]
calculating the formation channels for this dataset...
  Found 1922 BNS systems
Processing fiducial -> /Users/sashalvna/Research/Fit_SFRD_TNG/data/BNS/vanSon22/fiducial
  Saved 1922 systems with columns: ['dco_mass_1', 'dco_mass_2', 'delay_time', 'formation_channel', 'formation_efficiency_per_solar_mass', 'metallicity']
HDF5 file created: /Users/sashalvna/Research/Fit_SFRD_TNG/data/BNS/vanSon22/fiducial/bps_output.h5



In [11]:
# # New directory structure approach:
# BASE_DIR = '/Volumes/GROWL/GROWL_bps_compact/CHE'
# dco_type='BBH'
# filename_add = False
# CHE_mask = 'only CHE'

# # Process multiple Boesky24 models
# for dataset in boesky_dataset_list[:]: 
#     boesky_data = process_multiple_models(growl_catalog, 'Boesky24', [dataset],dco_type=dco_type, pessimistic= True, merges_hubble=True, no_RLOF_post_CE=True, CHE_mask=CHE_mask)
#     # Create separate entries for each dataset
#     output_files = create_hdf5_from_boesky_data(boesky_data, BASE_DIR, create_separate_files=True, filename_add=filename_add)
#     print()



## add Optimistic 

Boesky data doesnt have optimistic, but save this code for datasets that maybe do.. (check) 

In [12]:
# # # New directory structure approach:
# # BASE_DIR = '/Volumes/GROWL/GROWL_bps_compact/BBH'
# # dco_type='BBH'
# # filename_add = False
# # pessimistic = True

# # # Process multiple Boesky24 models
# # for dataset in boesky_dataset_list[:2]: 
# #     boesky_data = process_multiple_models(growl_catalog, 'Boesky24', [dataset],dco_type=dco_type, pessimistic=pessimistic, merges_hubble=True, no_RLOF_post_CE=True, CHE_mask='no CHE')
# #     # Create separate entries for each dataset
# #     output_files = create_hdf5_from_boesky_data(boesky_data, BASE_DIR, create_separate_files=True, filename_add=filename_add,  artificial_model_addition=False)
# #     print('\n \n')




# # New directory structure approach:
# BASE_DIR = '/Volumes/GROWL/GROWL_bps_compact/BBH'
# dco_type='BBH'
# filename_add = False
# pessimistic = False

# # Process multiple Boesky24 models
# for dataset in boesky_dataset_list[:2]: 
#     boesky_data = process_multiple_models(growl_catalog, 'Boesky24', [dataset],dco_type=dco_type, pessimistic=pessimistic, merges_hubble=True, no_RLOF_post_CE=True, CHE_mask='no CHE')
#     # Create separate entries for each dataset
#     output_files = create_hdf5_from_boesky_data(boesky_data, BASE_DIR, create_separate_files=True, filename_add=filename_add,  artificial_model_addition="optimisticCE")
#     print('\n \n')
# # len dco_mask  4881951  sum =  1649874

    
    

### Check if it worked by reading in the data:


In [13]:
import pandas as pd

full_path = '/Users/sashalvna/Research/Fit_SFRD_TNG/data/BBH/vanSon22/fiducial/bps_output.h5'

# Read the dataset into a pandas DataFrame
df = pd.read_hdf(full_path, key="input_data")

# Inspect the available columns
print(df.columns)

# Extract metallicities
metallicities = df["metallicity"].values
print(metallicities[:10])   # show first 10 values

# read in the units 
units = pd.read_hdf(full_path, key="units").to_dict()
print(units["delay_time"])   # 'Myr'

formation_channel = df['formation_channel'].values

print(formation_channel)


Index(['dco_mass_1', 'dco_mass_2', 'delay_time', 'formation_channel',
       'formation_efficiency_per_solar_mass', 'metallicity'],
      dtype='object')
[0.00035266 0.00205154 0.00027211 0.00015366 0.00052405 0.00020985
 0.00063717 0.00095198 0.00039831 0.00749585]
Myr
[2 2 2 ... 1 2 2]


## check Boesky formation channels

print out the formation channels for the other channel 

In [ ]:



#     channels = np.zeros_like(seeds).astype(int)
#     channels[classic_mask] = 1
#     channels[only_stable_mask] = 2
#     channels[single_core_mask] = 3
#     channels[double_core_mask] = 4
#     channels[classic_caseA_mask] = 1 # classify case A same as original channel
#     channels[only_stable_caseA_mask] = 2 #  # classify case A same as original channel

#     # within the other channel, assign other with CE  as -1
#     other_channel_mask_CE = np.in1d(seeds, all_cee_seeds)
#     mask_other_with_CE = (channels==0) & (other_channel_mask_CE==1)
#     channels[mask_other_with_CE] = -1 
    
#     # within the other channel, assign other without CE  as -2
#     mask_other_without_CE = (channels==0) & (other_channel_mask_CE==0)
#     channels[mask_other_without_CE] = -2


# growl_catalog = build_growl_catalog()
dco_type='BBH'
base_path = '/Volumes/GROWL/GROWL_bps_compact/%s/'%dco_type
# Assuming you have the GROWL catalog from the previous artifact
growl_catalog = build_growl_catalog(base_path='/Volumes/GROWL/GROWL_bps_compact/%s/'%dco_type)
#print GROWL catalog possible entries
print_catalog_summary(growl_catalog)

vanSon_dataset_list = list_datasets(growl_catalog, 'vanSon22')
print(vanSon_dataset_list)


for dataset in vanSon_dataset_list[:]: 
    output_hdf5_filename = os.path.join(get_folder_path_convolution_output(growl_catalog, "vanSon22", dataset), "bps_output.h5")
    print(output_hdf5_filename)
    df = pd.read_hdf(output_hdf5_filename, key="input_data")
    fc = df['formation_channel'].values
    weight = df['formation_efficiency_per_solar_mass'].values
    print('-----------------------------------')
    print('n GW sources ', len(weight))
    print(dataset, 'no MT: ', np.sum(weight[fc==-1])/np.sum(weight))
    
    
    print('\n \n')
    
    
    
dco_type='BHNS'
base_path = '/Volumes/GROWL/GROWL_bps_compact/%s/'%dco_type
# Assuming you have the GROWL catalog from the previous artifact
growl_catalog = build_growl_catalog(base_path='/Volumes/GROWL/GROWL_bps_compact/%s/'%dco_type)
#print GROWL catalog possible entries
print_catalog_summary(growl_catalog)

vanSon_dataset_list = list_datasets(growl_catalog, 'vanSon22')
print(boesky_dataset_list)


for dataset in vanSon_dataset_list[:]: 
    output_hdf5_filename = os.path.join(get_folder_path_convolution_output(growl_catalog, "vanSon22", dataset), "bps_output.h5")
    print(output_hdf5_filename)
    df = pd.read_hdf(output_hdf5_filename, key="input_data")
    fc = df['formation_channel'].values
    weight = df['formation_efficiency_per_solar_mass'].values
    print('-----------------------------------')
    print('n GW sources ', len(weight))
    print(dataset, 'no MT: ', np.sum(weight[fc==-1])/np.sum(weight))
    
    
    print('\n \n')
    


dco_type='BNS'
base_path = '/Volumes/GROWL/GROWL_bps_compact/%s/'%dco_type
# Assuming you have the GROWL catalog from the previous artifact
growl_catalog = build_growl_catalog(base_path='/Volumes/GROWL/GROWL_bps_compact/%s/'%dco_type)
#print GROWL catalog possible entries
print_catalog_summary(growl_catalog)

vanSon_dataset_list = list_datasets(growl_catalog, 'vanSon22')
print(vanSon_dataset_list)


for dataset in vanSon_dataset_list[:]: 
    output_hdf5_filename = os.path.join(get_folder_path_convolution_output(growl_catalog, "vanSon22", dataset), "bps_output.h5")
    print(output_hdf5_filename)
    df = pd.read_hdf(output_hdf5_filename, key="input_data")
    fc = df['formation_channel'].values
    weight = df['formation_efficiency_per_solar_mass'].values
    print('-----------------------------------')
    print('n GW sources ', len(weight))
    print(dataset, 'no MT: ', np.sum(weight[fc==-1])/np.sum(weight))
    
    
    print('\n \n')
    




GROWL Catalog Summary:

Author: Boesky24
  File: bps_output.h5
  Datasets (20):
    - alpha0_1beta0_25
    - alpha0_1beta0_5
    - alpha0_1beta0_75
    - alpha0_5beta0_25
    - alpha0_5beta0_5
    - alpha0_5beta0_75
    - alpha10_beta0_25
    - alpha10_beta0_5
    - alpha10_beta0_75
    - alpha2_beta0_5
    - alpha2_beta0_75
    - alpha2beta0_25
    - sigma_265_RMP_Mandel
    - sigma_265_RMP_Rapid
    - sigma_30_RMP_Delayed
    - sigma_30_RMP_Mandel
    - sigma_30_RMP_Rapid
    - sigma_750_RMP_Delayed
    - sigma_750_RMP_Mandel
    - sigma_750_RMP_Rapid
['alpha0_1beta0_25', 'alpha0_1beta0_5', 'alpha0_1beta0_75', 'alpha0_5beta0_25', 'alpha0_5beta0_5', 'alpha0_5beta0_75', 'alpha10_beta0_25', 'alpha10_beta0_5', 'alpha10_beta0_75', 'alpha2_beta0_5', 'alpha2_beta0_75', 'alpha2beta0_25', 'sigma_265_RMP_Mandel', 'sigma_265_RMP_Rapid', 'sigma_30_RMP_Delayed', 'sigma_30_RMP_Mandel', 'sigma_30_RMP_Rapid', 'sigma_750_RMP_Delayed', 'sigma_750_RMP_Mandel', 'sigma_750_RMP_Rapid']
/Volumes/GROWL/GROW

-----------------------------------
n GW sources  29964
alpha10_beta0_5    other with CE:  0.0038606020627591025
alpha10_beta0_5 other without CE:  0.0

 

/Volumes/GROWL/GROWL_bps_compact/BHNS/Boesky24/alpha10_beta0_75/bps_output.h5
-----------------------------------
n GW sources  34196
alpha10_beta0_75    other with CE:  0.0020579511800176697
alpha10_beta0_75 other without CE:  0.0

 

/Volumes/GROWL/GROWL_bps_compact/BHNS/Boesky24/alpha2_beta0_5/bps_output.h5
-----------------------------------
n GW sources  20127
alpha2_beta0_5    other with CE:  0.04882201149391087
alpha2_beta0_5 other without CE:  0.0

 

/Volumes/GROWL/GROWL_bps_compact/BHNS/Boesky24/alpha2_beta0_75/bps_output.h5
-----------------------------------
n GW sources  20344
alpha2_beta0_75    other with CE:  0.019753276917404188
alpha2_beta0_75 other without CE:  0.0

 

/Volumes/GROWL/GROWL_bps_compact/BHNS/Boesky24/alpha2beta0_25/bps_output.h5
-----------------------------------
n GW sources  52700
alpha2beta0_25  

In [12]:
# more basic only using hdf5 (but pandas might be better)

# def create_convolution_hdf5_from_dco_data(
#     dco_data: Dict[str, DCOParameters], 
#     base_output_dir: str,
#     author_name: str = "Boesky24",
#     output_filename: str = "output_example.h5",
#     create_separate_files: bool = False
# ) -> Dict[str, str]:
#     output_files = {}
    
#     if create_separate_files:
#         for dataset_name, dco_params in dco_data.items():
#             output_dir = os.path.join(base_output_dir, author_name, dataset_name)
#             os.makedirs(output_dir, exist_ok=True)
            
#             output_hdf5_filename = os.path.join(output_dir, output_filename)
#             print(f"Processing {dataset_name} -> {output_dir}")
            
#             # Open HDF5 file for writing
#             with h5py.File(output_hdf5_filename, "w") as f:
#                 grp = f.create_group("input_data")
                
#                 grp.create_dataset("delay_time", data=dco_params.delay_times)
#                 grp.create_dataset("metallicity", data=dco_params.metallicities)
#                 grp.create_dataset("formation_efficiency_per_solar_mass", data=dco_params.formation_efficiencies)
#                 grp.create_dataset("dco_mass_1", data=dco_params.dco_masses_1)
#                 grp.create_dataset("dco_mass_2", data=dco_params.dco_masses_2)
            
#             print(f"  Saved {dco_params.n_systems} systems")
#             output_files[dataset_name] = output_hdf5_filename
#             print(f"HDF5 file created: {output_hdf5_filename}")
    
#     return output_files

In [13]:
# # some code for quick check 
# full_path = '/Volumes/GROWL/GROWL_bps_compact/BBH/Boesky24/alpha0_1beta0_25/bps_output.h5'

# df = pd.read_hdf(full_path, key="input_data")

# # Inspect the available columns
# print(df.columns)

# fc = df['formation_channel']

